# MobileNetV3（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，MobileNetV3を用いてCIFAR100データセットの100クラス物体認識を行う．MobileNetV2のInverted Residual構造にSqueeze-and-Excitation（SE）ブロックとh-swish活性化関数を組み合わせることで，軽量なネットワークの認識精度をさらに改善する方法について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100は32×32ピクセルのカラー画像100クラスからなるデータセットです．学習用データには`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## MobileNetV3
MobileNetV3は，`mobilenet_v2.ipynb`で実装したInverted Residual構造（1×1畳み込みでチャンネル数を拡張し，Depthwise Convolutionを適用したのち，1×1畳み込みで削減するブロック）をベースに，次の2つの工夫を追加したネットワークです．

1. 一部のブロックに，チャンネルごとの重要度を学習するSqueeze-and-Excitation（SEブロック）を追加する．
2. Sigmoid・Swishの代わりに，計算コストの低い近似関数である**h-sigmoid・h-swish**を活性化関数として用いる．

### SEブロックの追加位置
SEブロックの仕組み自体は`senet.ipynb`で説明した通りです．MobileNetV3では，SEブロックをDepthwise Convolutionの出力（Projectionで削減する前の，拡張されたチャンネル数を持つ特徴マップ）に適用し，ゲート関数にはSigmoidではなく後述するh-sigmoidを用います．また，削減率はブロックごとに拡張後のチャンネル数に対して4分の1に固定します．

### h-swish・h-sigmoid
Swish（$x \cdot \text{sigmoid}(x)$）はReLUより高い認識精度が得られる活性化関数として知られていますが，Sigmoidの計算はモバイル端末上でのコストが大きいという欠点があります．そこでMobileNetV3では，Sigmoidを区分線形関数のReLU6で近似したh-sigmoidと，それを用いたh-swishを提案しています．

$$\text{h-sigmoid}(x) = \frac{\text{ReLU6}(x + 3)}{6}, \qquad \text{h-swish}(x) = x \cdot \text{h-sigmoid}(x)$$

ReLU6は多くの推論ライブラリ・ハードウェアで高速に計算できるため，Swish・Sigmoidとほぼ同等の形状を保ちながら計算コストを抑えることができます．

### CIFAR100への入力サイズの適応
MobileNetV3にも，224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR100のような小さな入力を対象とした「CIFAR版」があります．オリジナルのMobileNetV3-Smallはstemと複数のBneckブロックのstride=2によって特徴マップを大きく縮小しますが，32×32の入力にそのまま適用すると特徴マップが早期に消失してしまいます．そのため，本ノートブックでは`mobilenet_v2.ipynb`と同様にstemのstrideとstride=2となる段数を減らしたCIFAR版を実装します（詳細は本ノートブック末尾の「ImageNet版MobileNetV3とCIFAR版（本ノートブック）の違い」を参照）．

In [ ]:
class HSigmoid(nn.Module):
    def forward(self, x):
        return F.relu6(x + 3., inplace=True) / 6.


class HSwish(nn.Module):
    def forward(self, x):
        return x * F.relu6(x + 3., inplace=True) / 6.


class SEBlock(nn.Module):
    def __init__(self, channel, reduction=4):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            HSigmoid(),  # SENetではSigmoidだが，MobileNetV3ではh-sigmoidをゲート関数に用いる
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        w = self.avgpool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

### Bneckブロック
上記のSEブロックとh-swishを組み込んだInverted Residual Blockを，MobileNetV3の論文にならって「Bneckブロック」と呼びます．拡張後のチャンネル数`exp_ch`が入力チャンネル数`in_ch`と等しい場合は，拡張のための1×1畳み込みを省略します．また，ブロックごとにSEブロックの有無（`use_se`）と活性化関数（`use_hs`：h-swishかReLUか）を切り替えられるようにします．ショートカット接続は，Inverted Residualと同様にstrideが1かつ入出力のチャンネル数が等しい場合のみ行います．

In [ ]:
class Bneck(nn.Module):
    def __init__(self, in_ch, exp_ch, out_ch, kernel_size, stride, use_se, use_hs):
        super().__init__()
        assert stride in (1, 2)
        self.use_res = stride == 1 and in_ch == out_ch
        act = HSwish() if use_hs else nn.ReLU(inplace=True)

        layers = []
        if exp_ch != in_ch:
            # 1x1畳み込みでチャンネル数をexp_chに拡張
            layers += [nn.Conv2d(in_ch, exp_ch, kernel_size=1, bias=False),
                       nn.BatchNorm2d(exp_ch),
                       act]
        layers += [
            # Depthwise Convolution
            nn.Conv2d(exp_ch, exp_ch, kernel_size=kernel_size, stride=stride,
                      padding=kernel_size // 2, groups=exp_ch, bias=False),
            nn.BatchNorm2d(exp_ch),
            act,
        ]
        self.conv = nn.Sequential(*layers)
        self.se = SEBlock(exp_ch) if use_se else nn.Identity()
        # 1x1畳み込みでチャンネル数を削減（Projection）．活性化関数は適用しない
        self.project = nn.Sequential(
            nn.Conv2d(exp_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        out = self.conv(x)
        out = self.se(out)  # Depthwise Convolutionの出力にSEブロックを適用してからProjection
        out = self.project(out)
        if self.use_res:
            out = out + x
        return out

### ネットワークの定義
MobileNetV3-Smallは，Bneckブロックを下の表のように積み重ねて構成します．各行は，カーネルサイズ$k$，拡張後のチャンネル数`exp`，出力チャンネル数$c$，SEブロックの有無，活性化関数（RE: ReLU，HS: h-swish），各段のstride $s$を表します．

| k | exp | c | SE | NL | s |
|---|---|---|---|---|---|
| 3 | 16 | 16 | Y | RE | 1 |
| 3 | 72 | 24 | N | RE | 2 |
| 3 | 88 | 24 | N | RE | 1 |
| 5 | 96 | 40 | Y | HS | 1 |
| 5 | 240 | 40 | Y | HS | 1 |
| 5 | 240 | 40 | Y | HS | 1 |
| 5 | 120 | 48 | Y | HS | 1 |
| 5 | 144 | 48 | Y | HS | 1 |
| 5 | 288 | 96 | Y | HS | 2 |
| 5 | 576 | 96 | Y | HS | 1 |
| 5 | 576 | 96 | Y | HS | 1 |

最初の3×3畳み込み（stem）で入力画像を16チャンネルに変換したのち，上表のBneckブロックを順に適用します．最後に1×1畳み込みで576チャンネルに拡張し，Global Average Poolingで空間方向を1×1に集約した後，全結合層2つ（576→1024→クラス数）とh-swishからなる分類器でクラス数（100）に変換します．

In [ ]:
def _make_stage(in_ch, exp_ch, out_ch, kernel_size, stride, use_se, use_hs):
    return Bneck(in_ch, exp_ch, out_ch, kernel_size, stride, use_se, use_hs)


class MobileNetV3Small(nn.Module):
    # k, exp, c, se, hs, s
    cfg = [
        [3, 16, 16, True, False, 1],
        [3, 72, 24, False, False, 2],
        [3, 88, 24, False, False, 1],
        [5, 96, 40, True, True, 1],    # 元のImageNet版はs=2だが，CIFAR100 (32x32)向けにs=1へ変更
        [5, 240, 40, True, True, 1],
        [5, 240, 40, True, True, 1],
        [5, 120, 48, True, True, 1],
        [5, 144, 48, True, True, 1],
        [5, 288, 96, True, True, 2],
        [5, 576, 96, True, True, 1],
        [5, 576, 96, True, True, 1],
    ]

    def __init__(self, n_class=100):
        super().__init__()
        in_ch = 16
        self.stem = nn.Sequential(
            nn.Conv2d(3, in_ch, kernel_size=3, stride=1, padding=1, bias=False),  # CIFAR100向けにstride=1
            nn.BatchNorm2d(in_ch),
            HSwish(),
        )

        blocks = []
        for k, exp_ch, c, se, hs, s in self.cfg:
            blocks.append(_make_stage(in_ch, exp_ch, c, k, s, se, hs))
            in_ch = c
        self.blocks = nn.Sequential(*blocks)

        last_exp = 576
        self.conv_head = nn.Sequential(
            nn.Conv2d(in_ch, last_exp, kernel_size=1, bias=False),
            nn.BatchNorm2d(last_exp),
            HSwish(),
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(last_exp, 1024),
            HSwish(),
            nn.Linear(1024, n_class),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

## ネットワークの作成
MobileNetV3-Smallを作成し，`.to(device)`によりモデルのパラメータを`device`上に配置します．最適化手法にはモーメンタムSGD（学習率0.01，モーメンタム0.9，weight decay 5e-4）を用います．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = MobileNetV3Small(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版MobileNetV3とCIFAR版（本ノートブック）の違い

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| 入力画像サイズ | 224×224 | 32×32 |
| stemの畳み込みのstride | 2 | 1 |
| ブロック内でstride=2となる段数 | 4段（stem含め計5回のダウンサンプリング） | 2段（stem含め計3回のダウンサンプリング） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

元のMobileNetV3-Smallは，stemと複数のBneckブロックのstride=2によって入力を大きく縮小する設計になっており，224×224の入力を最終的に7×7程度まで縮小します．CIFAR100は32×32と入力サイズが小さいため，同じ回数だけダウンサンプリングすると特徴マップがすぐに1×1以下になってしまいます．そこで本ノートブックでは，`mobilenet_v2.ipynb`と同様にstemのstrideを1にし，stage表のうち1段（$c=40$の段）のstrideを2から1に変更することで，ダウンサンプリングの回数を減らし，最終的な特徴マップサイズを8×8としています．

## 課題

1. 各Bneckブロックの`use_se`を全て`False`にしてSEブロックを取り除いたネットワークを学習し，SEブロックの有無で認識精度や学習曲線がどのように異なるか比較しましょう．

2. `use_hs`を全て`False`にしてh-swishの代わりに全てReLUを用いた場合と比較し，活性化関数の違いによる認識精度の変化を確認しましょう．

3. `cfg`のstage表（カーネルサイズ，拡張後のチャンネル数，SEブロックの有無など）を変更して，パラメータ数や認識精度がどのように変化するか確認しましょう．